In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import scipy
import skimage

In [ ]:
theory_file = r"C:\DATA\4ranges-volume-momspe-2d-sw.dat"
data = []
with open(theory_file) as f:
    for line in f:
        parts = line.strip().split("  ")
        if len(parts) == 6:
            data.append([float(x) for x in parts])
data = np.asarray(data)
data = data[data[:, 0] < 1]


def fit_fn(theta, theta0, a, b):
    """Cosine fitting function for angular data."""
    return a * np.cos((theta - theta0) * 2) + b


# Extract data components
pr = data[:, 0]  # Momentum transfer values
theta = data[:, 1] % (2 * np.pi) - np.pi  # Angle values normalized to [-π, π]
intensity = data[:, -1]  # Intensity values

# Reshape to 2D grid based on unique values
grid_size = (len(set(pr)), len(set(theta)))
pr = pr.reshape(grid_size)
theta = theta.reshape(grid_size)
intensity = intensity.reshape(grid_size)
r_values = np.asarray(sorted(np.unique(pr)))
theta_values = np.asarray(sorted(np.unique(theta)))

# Interpolate onto regular grid with 1024 points
RESOLUTION = 1024
grid_indices = (np.linspace(0, len(r_values), RESOLUTION),
                np.linspace(0, len(theta_values), RESOLUTION))
rr, tt = np.meshgrid(*grid_indices)
intensity = scipy.ndimage.map_coordinates(intensity, [rr.flatten(), tt.flatten()])

# Create physical coordinate meshgrid
r_values = np.linspace(min(r_values), max(r_values), RESOLUTION)
theta_values = np.linspace(min(theta_values), max(theta_values), RESOLUTION)
pr, theta = np.meshgrid(r_values, theta_values)
intensity = intensity.reshape(RESOLUTION, RESOLUTION).T

# fill zero rows and columns by copying from the prior index
for i in range(intensity.shape[0]):
    if np.all(intensity[i, :] == 0) and i > 0:
        intensity[i, :] = intensity[i - 1, :]
for j in range(intensity.shape[1]):
    if np.all(intensity[:, j] == 0) and j > 0:
        intensity[:, j] = intensity[:, j - 1]

I_theory = intensity / np.max(intensity)  # Normalize intensity
pr_theory = r_values
theta_theory = theta_values

del data, r_values, theta_values, grid_indices, rr, tt, intensity, pr, theta, line, f, parts, theory_file, grid_size

#plot the theory data
px.imshow(
        I_theory,
        y=pr_theory,
        x=theta_theory,
        color_continuous_scale='inferno',
        labels={'y': 'Momentum Transfer (a.u.)', 'x': 'Angle (rad)', 'color': 'Intensity'},
        title='Theoretical Intensity Distribution',
        aspect='auto',
        origin='lower'
).show()

In [ ]:
fname_exp = r"J:\ctgroup\Edward\DATA\VMI\20220613\xe005_e_calibrated.h5"
base_data = pd.read_hdf(fname_exp, key="data")

# Filter and symmetrize
data = (base_data
        .query("sqrt(px**2+py**2+pz**2)<1")
        .query("abs(py)<0.5")
        .query("pz>0")
        .reset_index(drop=True))
data_sym = data.copy()
data_sym[['px', 'py', 'pz']] *= -1
data = pd.concat([data, data_sym], ignore_index=True)

# Compute spherical coordinates
data['pr'] = np.linalg.norm(data[['px', 'py', 'pz']].values, axis=1)
data['phi'] = np.arctan2(data['px'], data['pz'])
data['theta'] = np.arccos(data['px'] / data['pr'])
data['E'] = data['pr'] ** 2 / 2

# Build 2D histogram
RES = 1024
h2d, re_edges, pe_edges = np.histogram2d(data['pr'], data['phi'], bins=RES)
r_vals = 0.5 * (re_edges[:-1] + re_edges[1:])
theta_vals = 0.5 * (pe_edges[:-1] + pe_edges[1:])

# Resape histogram to 2D grid
grid_indices = (np.linspace(0, len(r_vals), RESOLUTION),
                np.linspace(0, len(theta_vals), RESOLUTION))
rr, tt = np.meshgrid(*grid_indices)
h2d = scipy.ndimage.map_coordinates(h2d, [rr.flatten(), tt.flatten()])

# Create physical coordinate meshgrid
r_values = np.linspace(min(r_vals), max(r_vals), RESOLUTION)
theta_values = np.linspace(min(theta_vals), max(theta_vals), RESOLUTION)
pr, theta = np.meshgrid(r_values, theta_values)
h2d = h2d.reshape(RESOLUTION, RESOLUTION).T

I_exp = h2d / np.max(h2d)  # Normalize intensity
pr_exp = r_values
theta_exp = theta_values

# Clean up unused variables
del base_data, data, data_sym, h2d, re_edges, pe_edges, r_vals, theta_vals, grid_indices, rr, tt, \
    r_values, theta_values, pr, theta
# Plot experimental data
px.imshow(
        I_exp,
        y=pr_exp,
        x=theta_exp,
        color_continuous_scale='inferno',
        labels={'y': 'Momentum Transfer (a.u.)', 'x': 'Angle (rad)', 'color': 'Intensity'},
        title='Experimental Intensity Distribution',
        aspect='auto',
        origin='lower',
).show()

In [ ]:
# New cell: radial distributions + smoothing + peak markings
import numpy as np
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
import plotly.graph_objects as go

# Compute radial distributions (mean over theta)
radial_th = np.mean(I_theory, axis=1)
radial_exp = np.mean(I_exp, axis=1)

# Smooth experimental radial distribution
radial_exp_smooth = gaussian_filter1d(radial_exp, sigma=2)

# Normalize both curves
radial_th_norm = radial_th / np.max(radial_th)
radial_exp_norm = radial_exp_smooth / np.max(radial_exp_smooth)

# Detect peaks and select top 3 by height
peaks_th, _ = find_peaks(radial_th_norm, distance=20)
top3_th = np.sort(peaks_th[np.argsort(radial_th_norm[peaks_th])[-4:]])

peaks_exp, _ = find_peaks(radial_exp_norm, distance=20)
top3_exp = np.sort(peaks_exp[np.argsort(radial_exp_norm[peaks_exp])[-4:]])

# Plot with Plotly
fig = go.Figure()
fig.add_trace(go.Scatter(x=pr_theory, y=radial_th_norm,
                         name='Theory', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=pr_exp, y=radial_exp_norm,
                         name='Experiment (smoothed)', line=dict(color='red')))
fig.add_trace(go.Scatter(x=pr_theory[top3_th], y=radial_th_norm[top3_th],
                         mode='markers', marker=dict(color='blue', symbol='x', size=10),
                         name='Theory Peaks'))
fig.add_trace(go.Scatter(x=pr_exp[top3_exp], y=radial_exp_norm[top3_exp],
                         mode='markers', marker=dict(color='red', symbol='circle', size=10),
                         name='Exp Peaks'))

# Compute metric annotations
peak_pr_th = pr_theory[top3_th]
peak_pr_exp = pr_exp[top3_exp]
ratio_peaks = peak_pr_exp / peak_pr_th
# Energies in eV (E = pr^2/2 in Hartree ×27.2114)
E_th = peak_pr_th ** 2 / 2 * 27.2114
E_exp = peak_pr_exp ** 2 / 2 * 27.2114
# Energy spacings
dE_th = np.diff(E_th)
dE_exp = np.diff(E_exp)
# Wavelength spacings in nm (λ ≈1240 eV·nm / ΔE)
lambda_th = 1240.0 / dE_th
lambda_exp = 1240.0 / dE_exp
# Build annotation text
annotation_text = (
    f"Peak ratios exp/th: {ratio_peaks.round(3)}\n"
    f"E spacings (th) eV: {dE_th.round(2)}\n"
    f"λ spacings (th) nm: {lambda_th.round(1)}\n"
    f"E spacings (exp) eV: {dE_exp.round(2)}\n"
    f"λ spacings (exp) nm: {lambda_exp.round(1)}"
)
# Print the annotations instead of adding them to the figure
print(annotation_text)

fig.update_layout(title='Radial Distributions with Peaks',
                  xaxis_title='pr (a.u.)', yaxis_title='Normalized Intensity')
fig.show()


In [ ]:
# New cell: least squares fit for pr scaling
from scipy.optimize import least_squares
from scipy.ndimage import gaussian_filter1d

# Pre-smooth the normalized experimental radial distribution
radial_exp_smoothed = gaussian_filter1d(radial_exp_norm, sigma=2)


def residual(alpha):
    scaled_pr = pr_exp * alpha
    exp_interp = np.interp(pr_theory, scaled_pr, radial_exp_smoothed, left=0, right=0)
    # use full pr range instead of window
    return exp_interp - radial_th_norm


res = least_squares(residual, x0=1.0)
alpha_opt = res.x[0]
print(f"Optimal scaling factor α = {alpha_opt:.4f}")

# Plot comparison after scaling (use the smoothed curve)
scaled_radial_exp = np.interp(pr_theory, pr_exp * alpha_opt, radial_exp_smoothed, left=0, right=0)
import plotly.graph_objects as go

fig = go.Figure([
    go.Scatter(x=pr_theory, y=radial_th_norm, name='Theory', line=dict(color='blue')),
    go.Scatter(x=pr_theory, y=scaled_radial_exp, name='Exp (scaled)',
               line=dict(color='red', dash='dash')),
    go.Scatter(x=pr_theory[top3_th], y=radial_th_norm[top3_th],
               mode='markers', marker=dict(color='blue', symbol='x', size=10),
               name='Theory Peaks'),
    go.Scatter(x=pr_exp[top3_exp] * alpha_opt, y=radial_exp_norm[top3_exp],
               mode='markers', marker=dict(color='red', symbol='circle', size=10),
               name='Exp Peaks')
])
fig.update_layout(
        title=f'Least Squares Fit (α={alpha_opt:.4f})',
        xaxis_title='pr (a.u.)',
        yaxis_title='Normalized Intensity'
)
fig.show()


In [ ]:
# New cell: optimal angular shift of experimental map via phi‐axis roll
from scipy.optimize import least_squares
from scipy.interpolate import interp1d

# compute φ step size
dphi = theta_exp[1] - theta_exp[0]

# Precompute scaled and smoothed experimental map onto theory pr grid
f_map = interp1d(pr_exp * alpha_opt, I_exp, axis=0,
                 bounds_error=False, fill_value=0.0)
I_exp_scaled_on_th = f_map(pr_theory)
# Normalize and smooth along phi axis
I_theory_norm = I_theory / np.max(I_theory)
I_exp_scaled_smoothed = skimage.filters.gaussian(
        I_exp_scaled_on_th, sigma=10, mode='wrap', preserve_range=True
)
# Normalize the smoothed experimental map

# Determine the first radial‐distribution peak index
first_peak_idx = top3_th[0]

# define angular domain for periodic wrap
theta_min = theta_exp.min()
period = theta_exp.max() - theta_min


def shifted_exp(phi0):
    # shift via interpolation along phi (θ) axis with periodic wrap
    new_theta = theta_exp - phi0
    # wrap angles into [theta_min, theta_min+period)
    new_theta = ((new_theta - theta_min) % period) + theta_min
    from scipy.interpolate import interp1d
    f_phi = interp1d(theta_exp,
                     I_exp_scaled_smoothed,
                     axis=1,
                     bounds_error=False,
                     fill_value='extrapolate')
    return f_phi(new_theta)


# compute widths of all theory peaks at half‐max
from scipy.signal import peak_widths

widths_th, width_heights, left_ips, right_ips = peak_widths(
        radial_th_norm, peaks_th, rel_height=0.1
)
# locate the first peak and its window bounds
first_peak_idx = top3_th[0]
idx0 = np.where(peaks_th == first_peak_idx)[0][0]
start_idx = int(np.floor(left_ips[idx0]))
end_idx = int(np.ceil(right_ips[idx0]))


def residual_phi(phi0):
    diff = shifted_exp(phi0) - I_theory_norm
    # only use rows within the first peak width
    window = diff[start_idx:end_idx + 1, :]
    return window.ravel()


# optimize using only that radial window
res_phi = least_squares(residual_phi, x0=np.pi - 2.62941 - np.radians(6))
phi_opt = res_phi.x[0]
print(f"Optimal φ shift: {phi_opt:.4f} rad ({np.degrees(phi_opt):.2f}°)")
print(f"Fit window rows: {start_idx}–{end_idx}, residual norm: {np.linalg.norm(res_phi.fun):.4f}")

# quick visual check at first radial peak
exp_shift_on_th = shifted_exp(phi_opt)
import plotly.graph_objects as go

fig = go.Figure([
    go.Scatter(x=theta_theory, y=I_theory_norm[first_peak_idx, :], name='Theory'),
    go.Scatter(x=theta_exp, y=exp_shift_on_th[first_peak_idx, :], name='Exp shifted')
])
fig.update_layout(
        title='Angular Alignment at First Radial Peak',
        xaxis_title='Angle (rad)',
        yaxis_title='Normalized Intensity'
)
fig.show()

print(np.degrees(np.pi - 2.62941))

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# extract the radial window used in fitting
pr_fit = pr_theory[start_idx:end_idx + 1]
region_th = I_theory_norm[start_idx:end_idx + 1, :]
region_exp = exp_shift_on_th[start_idx:end_idx + 1, :]

# side-by-side heatmaps
fig = make_subplots(rows=2, cols=1,
                    subplot_titles=['Theory Fit Region', 'Exp Fit Region'])
fig.add_trace(go.Heatmap(z=region_th, x=theta_theory, y=pr_fit,
                         colorscale='inferno', showscale=False),
              row=1, col=1)
fig.add_trace(go.Heatmap(z=region_exp, x=theta_exp, y=pr_fit,
                         colorscale='inferno', showscale=False),
              row=2, col=1)
fig.update_layout(title='Angular Fitting Region', height=800, width=800)
fig.show()
